# Candidate Enrichment & Deep Ranking Pipeline

This notebook contains the complete, production-ready candidate enrichment and deep ranking pipeline for the **Redrob AI Challenge**. It is designed to run end-to-end on Google Colab with GPU acceleration (optional but highly recommended for fast embedding and Cross-Encoder inference).

### Architecture Overview
- **Phase 1: Narrative Synthesis**: Enrich raw profiles into clean natural language narratives.
- **Phase 2: Precomputation & LTR Training**: Build BM25 index, generate dense BGE embeddings, engineer behavioral feature arrays, build interaction terms, detect honeypots, and train an XGBoost Learning-to-Rank (LTR) model.
- **Phase 3 & 4: Retrieval and Ranking**: Combine BM25 & BGE similarity rankings using Reciprocal Rank Fusion (RRF), score candidates with XGBoost LTR, refine the top 50 with Cross-Encoder re-ranking, and generate verified reasoning for the top 100.

### Dataset Placement
Make sure to place the dataset directory `India_runs_data_and_ai_challenge` directly inside `/content` (i.e. `/content/India_runs_data_and_ai_challenge`).


## 1. Setup & Installation
Install the required libraries. This cell will automatically download and install `sentence-transformers`, `rank-bm25`, `xgboost`, and other necessary libraries.


In [ ]:
!pip install --upgrade pip
!pip install numpy pandas scikit-learn rank-bm25 sentence-transformers torch tqdm xgboost


## 2. Imports & Configuration
Configure imports and define input/output paths.


In [ ]:
import os
import sys
import json
import pickle
import time
import re
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
import xgboost as xgb
import torch
from tqdm import tqdm

# ==============================================================================
# CONFIGURATION & PATH SETUP
# ==============================================================================
DATASET_DIR = '/content/India_runs_data_and_ai_challenge'

CANDIDATES_PATH = f'{DATASET_DIR}/candidates.jsonl'
JOB_DESCRIPTION_PATH = f'{DATASET_DIR}/job_description.docx'

PROCESSED_DIR = '/content/data/processed'
ARTIFACTS_DIR = '/content/data/artifacts'

ENRICHED_CANDIDATES_PATH = f'{PROCESSED_DIR}/enriched_candidates.jsonl'
BM25_INDEX_PATH = f'{ARTIFACTS_DIR}/bm25.pkl'
EMBEDDINGS_PATH = f'{ARTIFACTS_DIR}/embeddings.npy'
EMBEDDING_IDS_PATH = f'{ARTIFACTS_DIR}/embedding_ids.npy'
FEATURES_PATH = f'{ARTIFACTS_DIR}/features.npy'
FEATURES_META_PATH = f'{ARTIFACTS_DIR}/features_meta.npz'
FEATURES_IX_PATH = f'{ARTIFACTS_DIR}/features_ix.npy'
SILVER_LABELS_PATH = f'{ARTIFACTS_DIR}/silver_labels.csv'
HONEYPOT_FLAGS_PATH = f'{ARTIFACTS_DIR}/honeypot_flags.json'
XGBOOST_LTR_MODEL_PATH = f'{ARTIFACTS_DIR}/xgboost_ltr.model'

OUTPUT_SUBMISSION_PATH = '/content/submission.csv'

# Create directories
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)


## 3. Phase 1: Text Enrichment
Ported from `src/phase1_enrichment.py`. This synthesizes candidate career history, skills, and engagement signals into structured narratives.


In [ ]:
def build_career_narrative(profile, career_history):
    lines = []
    headline = profile.get('headline', '')
    summary = profile.get('summary', '')
    yoe = profile.get('years_of_experience', 0)
    lines.append(f'Candidate is a {headline} with {yoe} years of experience.')
    if summary:
        lines.append(f'Professional Summary: {summary}')
        
    lines.append('Career History:')
    for role in career_history:
        company = role.get('company', 'Unknown Company')
        title = role.get('title', 'Unknown Title')
        duration = role.get('duration_months', 0)
        desc = role.get('description', '')
        
        lines.append(f'- Worked at {company} as {title} for {duration} months. {desc}')
        
    return '\n'.join(lines)

def build_skill_narrative(skills):
    if not skills:
        return 'No skills listed.'
        
    lines = ['Skills:']
    for skill in skills:
        name = skill.get('name', 'Unknown Skill')
        proficiency = skill.get('proficiency', 'unknown proficiency').capitalize()
        duration = skill.get('duration_months', 0)
        endorsements = skill.get('endorsements', 0)
        
        lines.append(f'- {proficiency} {name} - {duration} months used, {endorsements} endorsements')
        
    return '\n'.join(lines)

def build_behavioral_text(signals):
    if not signals:
        return 'No behavioral signals available.'
        
    lines = ['Behavioral Signals:']
    
    last_active = signals.get('last_active_date', 'Unknown')
    lines.append(f'- Last active on {last_active}.')
    
    rr = signals.get('recruiter_response_rate', 0)
    lines.append(f'- Responds to {int(rr * 100)}% of recruiter messages.')
    
    github = signals.get('github_activity_score', -1)
    if github >= 0:
        lines.append(f'- GitHub activity score is {github}/100.')
    else:
        lines.append('- No GitHub account linked.')
        
    np = signals.get('notice_period_days', 0)
    lines.append(f'- Notice period is {np} days.')
    
    views = signals.get('profile_views_received_30d', 0)
    apps = signals.get('applications_submitted_30d', 0)
    lines.append(f'- Received {views} profile views and submitted {apps} applications in the last 30 days.')
    
    salary = signals.get('expected_salary_range_inr_lpa', {})
    if salary:
        lines.append(f'- Expected salary range is {salary.get("min", 0)} to {salary.get("max", 0)} LPA INR.')
        
    mode = signals.get('preferred_work_mode', 'any')
    relocate = 'willing' if signals.get('willing_to_relocate') else 'unwilling'
    lines.append(f'- Prefers {mode} work and is {relocate} to relocate.')
    
    return '\n'.join(lines)

def process_candidate(candidate):
    cid = candidate.get('candidate_id', 'UNKNOWN_ID')
    profile = candidate.get('profile', {})
    career = candidate.get('career_history', [])
    skills = candidate.get('skills', [])
    signals = candidate.get('redrob_signals', {})
    
    career_text = build_career_narrative(profile, career)
    skill_text = build_skill_narrative(skills)
    behavioral_text = build_behavioral_text(signals)
    
    enriched_text = f'--- CAREER NARRATIVE ---\n{career_text}\n\n--- SKILL NARRATIVE ---\n{skill_text}\n\n--- BEHAVIORAL SIGNALS ---\n{behavioral_text}'
    
    return {
        'candidate_id': cid,
        'enriched_text': enriched_text
    }

def run_phase1_enrichment(input_path, output_path):
    print(f'Phase 1: Enriching candidates from {input_path}...')
    if not os.path.exists(input_path):
        raise FileNotFoundError(f'Candidates file not found at {input_path}')
        
    count = 0
    with open(output_path, 'w', encoding='utf-8') as outfile:
        with open(input_path, 'r', encoding='utf-8') as infile:
            first_char = infile.read(1)
            infile.seek(0)
            
            if first_char == '[':
                candidates = json.load(infile)
                for candidate in candidates:
                    enriched = process_candidate(candidate)
                    outfile.write(json.dumps(enriched) + '\n')
                    count += 1
            else:
                for line in infile:
                    if not line.strip(): continue
                    candidate = json.loads(line)
                    enriched = process_candidate(candidate)
                    outfile.write(json.dumps(enriched) + '\n')
                    count += 1
                    
    print(f'Finished Phase 1: Enriched {count} records. Output written to {output_path}')


## 4. Phase 2: Offline Precomputations
Ported from `src/phase2_precompute/...`. This cell handles the parallelized embedding generation, feature processing, silver label compilation, honeypot detection, and LTR model training.


In [ ]:
JD_POSITIVE_TERMS = {
    'ai', 'ml', 'machine learning', 'deep learning', 'nlp', 'llm', 'rag', 'retrieval',
    'ranking', 'recommendation', 'search', 'semantic search', 'embeddings', 'vector',
    'faiss', 'elasticsearch', 'opensearch', 'pinecone', 'weaviate', 'qdrant', 'milvus',
    'python', 'pytorch', 'tensorflow', 'xgboost', 'evaluation', 'ndcg', 'map', 'mrr',
    'ab testing', 'a/b', 'production', 'deployed', 'product', 'backend', 'data pipeline'
}

BAD_TITLE_TERMS = {
    'mechanical engineer', 'civil engineer', 'graphic designer', 'sales executive',
    'marketing manager', 'accountant', 'hr manager', 'customer support', 'content writer'
}

GOOD_TITLE_TERMS = {
    'ai engineer', 'machine learning engineer', 'ml engineer', 'data scientist',
    'applied scientist', 'search engineer', 'ranking engineer', 'recommendation engineer',
    'nlp engineer', 'backend engineer', 'data engineer', 'software engineer', 'staff engineer'
}

def read_docx_text(file_path):
    try:
        with zipfile.ZipFile(file_path) as docx:
            xml_content = docx.read('word/document.xml')
            tree = ET.fromstring(xml_content)
            namespaces = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}
            texts = [node.text for node in tree.findall('.//w:t', namespaces) if node.text]
            return ' '.join(texts)
    except Exception:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()

def load_candidate_records(input_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        first_char = f.read(1)
        f.seek(0)
        if first_char == '[':
            return json.load(f)
        return [json.loads(line) for line in f if line.strip()]

def minmax(values):
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return arr
    lo = np.nanmin(arr)
    hi = np.nanmax(arr)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi - lo < 1e-12:
        return np.zeros_like(arr, dtype=float)
    return (arr - lo) / (hi - lo)

def safe_log_norm(values, cap=None):
    arr = np.log1p(np.maximum(np.asarray(values, dtype=float), 0.0))
    denom = np.log1p(cap) if cap else max(float(np.nanmax(arr)), 1.0)
    return np.clip(arr / max(denom, 1e-9), 0.0, 1.0)

def token_count(text, terms):
    text = (text or '').lower()
    return sum(1 for term in terms if term in text)

def compute_profile_fit_score(record):
    profile = record.get('profile', {})
    title = profile.get('current_title', '').lower()
    headline = profile.get('headline', '').lower()
    summary = profile.get('summary', '').lower()
    industry = profile.get('current_industry', '').lower()
    yoe = float(profile.get('years_of_experience', 0) or 0)
    skills = [str(s.get('name', '')).lower() for s in record.get('skills', [])]
    skill_text = ' '.join(skills)
    career_text = ' '.join(str(role.get('description', '')) for role in record.get('career_history', [])).lower()
    full_text = ' '.join([title, headline, summary, industry, skill_text, career_text])

    title_score = 0.0
    if any(term in title for term in GOOD_TITLE_TERMS):
        title_score = 1.0
    elif any(term in title for term in ['backend', 'data', 'software', 'platform', 'search']):
        title_score = 0.65
    elif any(term in title for term in BAD_TITLE_TERMS):
        title_score = 0.0
    else:
        title_score = 0.25

    ai_hits = token_count(full_text, JD_POSITIVE_TERMS)
    skill_hits = token_count(skill_text, JD_POSITIVE_TERMS)
    career_hits = token_count(career_text, JD_POSITIVE_TERMS)

    term_score = min(1.0, ai_hits / 14.0)
    skill_score = min(1.0, skill_hits / 6.0)
    career_score = min(1.0, career_hits / 7.0)

    if 5 <= yoe <= 9:
        exp_score = 1.0
    elif 4 <= yoe < 5 or 9 < yoe <= 11:
        exp_score = 0.75
    elif 3 <= yoe < 4 or 11 < yoe <= 14:
        exp_score = 0.45
    else:
        exp_score = 0.15

    product_terms = ['software', 'fintech', 'e-commerce', 'product', 'saas', 'marketplace', 'platform']
    product_score = 1.0 if any(t in industry or t in full_text for t in product_terms) else 0.35

    bad_title_penalty = 0.35 if any(term in title for term in BAD_TITLE_TERMS) else 0.0
    score = (
        0.24 * title_score +
        0.22 * career_score +
        0.20 * term_score +
        0.14 * skill_score +
        0.12 * exp_score +
        0.08 * product_score -
        bad_title_penalty
    )
    return float(np.clip(score, 0.0, 1.0))

def run_track_a_bm25(input_file, output_file):
    print(f'Track A: Building BM25 index from {input_file}...')
    corpus = []
    candidate_ids = []

    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            record = json.loads(line)
            candidate_ids.append(record['candidate_id'])
            text = record['enriched_text'].lower()
            corpus.append(text.split())

    print(f'Loaded {len(corpus)} records. Initializing BM25Okapi...')
    bm25 = BM25Okapi(corpus)
    with open(output_file, 'wb') as f:
        pickle.dump({'bm25_index': bm25, 'candidate_ids': candidate_ids}, f)
    print(f'Track A BM25 index saved to {output_file}')

def run_track_b_embeddings(input_file, output_file, batch_size=128):
    print(f'Track B: Loading enriched texts from {input_file}...')
    texts = []
    candidate_ids = []

    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            record = json.loads(line)
            candidate_ids.append(record['candidate_id'])
            texts.append(record['enriched_text'])

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Track B: Loaded {len(texts)} records. Initializing BGE model on device: {device}...')

    bge_local_path = Path(output_file).parent / 'bge_model'
    ce_local_path = Path(output_file).parent / 'ce_model'

    if bge_local_path.exists():
        print(f'Loading cached BGE model from {bge_local_path}...')
        model = SentenceTransformer(str(bge_local_path), device=device)
    else:
        model = SentenceTransformer('BAAI/bge-small-en-v1.5', device=device)
        print(f'Caching BGE model locally to {bge_local_path}...')
        model.save(str(bge_local_path))

    if ce_local_path.exists():
        print(f'Loading cached Cross-Encoder from {ce_local_path}...')
        _ = CrossEncoder(str(ce_local_path), device=device)
    else:
        print('Initializing Cross-Encoder model (ms-marco-MiniLM-L-6-v2)...')
        ce_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device)
        print(f'Caching Cross-Encoder model locally to {ce_local_path}...')
        ce_model.save(str(ce_local_path))

    print('Encoding candidate narratives (using sentence-transformers GPU/CPU encoder)...')
    embeddings = model.encode(texts, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True)

    print(f'Saving embeddings of shape {embeddings.shape} to {output_file}...')
    np.save(output_file, embeddings)
    ids_path = Path(output_file).with_name('embedding_ids.npy')
    np.save(ids_path, np.array(candidate_ids))
    print('Track B Embeddings precomputed successfully!')

def run_track_c_features(input_file, output_file):
    print(f'Track C: Loading raw candidate structured data from {input_file}...')
    candidate_ids = []
    features_list = []

    feature_cols = [
        'profile_completeness_score', 'profile_views_received_30d', 'applications_submitted_30d',
        'recruiter_response_rate', 'avg_response_time_hours', 'connection_count', 'endorsements_received',
        'notice_period_days', 'github_activity_score', 'search_appearance_30d', 'saved_by_recruiters_30d',
        'interview_completion_rate', 'offer_acceptance_rate', 'verified_email', 'verified_phone',
        'linkedin_connected', 'open_to_work_flag', 'willing_to_relocate', 'salary_min', 'salary_max'
    ]

    for record in load_candidate_records(input_file):
        candidate_ids.append(record['candidate_id'])
        signals = record.get('redrob_signals', {})
        row = []
        for col in feature_cols[:18]:
            val = signals.get(col, 0)
            if isinstance(val, bool):
                val = 1.0 if val else 0.0
            elif val is None:
                val = 0.0
            row.append(float(val))
        salary = signals.get('expected_salary_range_inr_lpa', {}) or {}
        row.append(float(salary.get('min', 0.0)))
        row.append(float(salary.get('max', 0.0)))
        features_list.append(row)

    X = np.array(features_list, dtype=float)
    for i in range(X.shape[1]):
        col = X[:, i]
        mask = (col == -1.0)
        if mask.any():
            valid_vals = col[~mask]
            col[mask] = np.median(valid_vals) if len(valid_vals) > 0 else 0.0

    print(f'Extracted features matrix of shape {X.shape}. Saving to {output_file}...')
    np.save(output_file, X)
    meta_path = Path(output_file).with_name('features_meta.npz')
    np.savez(meta_path, columns=feature_cols, candidate_ids=candidate_ids)
    print('Track C Features generated successfully!')

def run_interaction_builder(features_file, output_file):
    print(f'Interaction Builder: Loading base features from {features_file}...')
    X = np.load(features_file)

    views = safe_log_norm(X[:, 1], cap=250)
    apps = safe_log_norm(X[:, 2], cap=40)
    response = np.clip(X[:, 3], 0.0, 1.0)
    response_speed = 1.0 - np.clip(X[:, 4] / 168.0, 0.0, 1.0)
    connections = safe_log_norm(X[:, 5], cap=2000)
    endorsements = safe_log_norm(X[:, 6], cap=500)
    low_notice = 1.0 - np.clip(X[:, 7] / 90.0, 0.0, 1.0)
    github = np.clip(X[:, 8] / 100.0, 0.0, 1.0)
    search = safe_log_norm(X[:, 9], cap=500)
    saved = safe_log_norm(X[:, 10], cap=100)
    interview = np.clip(X[:, 11], 0.0, 1.0)
    offer = np.clip(X[:, 12], 0.0, 1.0)
    verified = np.mean(np.clip(X[:, 13:16], 0.0, 1.0), axis=1)
    open_to_work = np.clip(X[:, 16], 0.0, 1.0)
    relocate = np.clip(X[:, 17], 0.0, 1.0)
    salary_mid = (X[:, 18] + X[:, 19]) / 2.0
    salary_fit = 1.0 - np.clip(np.abs(salary_mid - 45.0) / 45.0, 0.0, 1.0)

    interactions = np.column_stack([
        open_to_work * response,
        open_to_work * low_notice,
        response * response_speed,
        interview * response,
        offer * interview,
        github * endorsements,
        search * saved,
        views * saved,
        verified * response,
        relocate * low_notice,
        salary_fit * open_to_work,
        salary_fit * response,
        connections * endorsements,
        search * response,
        saved * response,
        apps * open_to_work,
        (1.0 - apps) * saved,
        github * search,
        github * interview,
        low_notice * response_speed,
        verified * open_to_work,
        verified * low_notice,
        views * response,
        search * open_to_work,
        saved * open_to_work,
        salary_fit * low_notice,
        salary_fit * relocate,
    ])

    print(f'Saving meaningful interaction features of shape {interactions.shape} to {output_file}...')
    np.save(output_file, interactions.astype(np.float32))
    print('Interaction Builder completed!')

def compute_bm25_scores_from_jd(artifacts_dir, jd_text):
    with open(f'{artifacts_dir}/bm25.pkl', 'rb') as f:
        bm25_data = pickle.load(f)
    cleaned_jd = re.sub(r'[^\w\s]', ' ', jd_text.lower())
    jd_tokens = cleaned_jd.split()
    return np.asarray(bm25_data['bm25_index'].get_scores(jd_tokens), dtype=float), np.array(bm25_data['candidate_ids'])

def compute_semantic_scores_from_jd(artifacts_dir, jd_text):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    bge_model = SentenceTransformer(str(Path(artifacts_dir) / 'bge_model'), device=device)
    jd_vector = bge_model.encode([jd_text], convert_to_numpy=True)[0]
    embeddings = np.load(f'{artifacts_dir}/embeddings.npy')
    dot = np.dot(embeddings, jd_vector)
    denom = np.linalg.norm(embeddings, axis=1) * max(np.linalg.norm(jd_vector), 1e-10)
    denom[denom == 0] = 1e-10
    return dot / denom

def run_silver_label_generator(features_meta_file, output_file, jd_path=JOB_DESCRIPTION_PATH,
                               artifacts_dir=ARTIFACTS_DIR, enriched_path=ENRICHED_CANDIDATES_PATH,
                               top_k=2000, ce_batch_size=32):
    print('Silver Label Generator: building labels from actual JD and Cross-Encoder scores...')
    meta = np.load(features_meta_file)
    candidate_ids = np.array(meta['candidate_ids'])
    jd_text = read_docx_text(jd_path)

    semantic_scores = compute_semantic_scores_from_jd(artifacts_dir, jd_text)
    bm25_scores, bm25_ids = compute_bm25_scores_from_jd(artifacts_dir, jd_text)
    if list(bm25_ids) != list(candidate_ids):
        raise ValueError('BM25 candidate_id order does not match features_meta candidate_id order.')

    semantic_ranks = np.empty_like(semantic_scores, dtype=float)
    semantic_ranks[np.argsort(-semantic_scores)] = np.arange(1, len(semantic_scores) + 1)
    bm25_ranks = np.empty_like(bm25_scores, dtype=float)
    bm25_ranks[np.argsort(-bm25_scores)] = np.arange(1, len(bm25_scores) + 1)
    retrieval_scores = rrf_score(semantic_ranks, bm25_ranks)

    shortlist_size = min(top_k, len(candidate_ids))
    shortlist_indices = np.argsort(-retrieval_scores)[:shortlist_size]
    shortlist_ids = set(candidate_ids[shortlist_indices])

    enriched_texts = {}
    with open(enriched_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            record = json.loads(line)
            cid = record['candidate_id']
            if cid in shortlist_ids:
                enriched_texts[cid] = record['enriched_text'][:2200]

    ordered_ids = candidate_ids[shortlist_indices]
    pairs = [(jd_text, enriched_texts.get(cid, '')) for cid in ordered_ids]
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    ce_model = CrossEncoder(str(Path(artifacts_dir) / 'ce_model'), device=device)
    try:
        ce_scores = ce_model.predict(pairs, batch_size=ce_batch_size, show_progress_bar=True)
    except TypeError:
        ce_scores = ce_model.predict(pairs, batch_size=ce_batch_size)
    ce_scores = np.asarray(ce_scores, dtype=float)

    features = np.load(f'{artifacts_dir}/features.npy')
    features_ix = np.load(f'{artifacts_dir}/features_ix.npy')
    availability = minmax(features_ix[:, 0])[shortlist_indices]
    combined = 0.72 * minmax(ce_scores) + 0.18 * minmax(semantic_scores[shortlist_indices]) + 0.10 * availability

    q1, q2, q3 = np.quantile(combined, [0.35, 0.70, 0.90])
    labels = np.zeros(len(combined), dtype=int)
    labels[combined >= q1] = 1
    labels[combined >= q2] = 2
    labels[combined >= q3] = 3

    df = pd.DataFrame({
        'candidate_id': ordered_ids,
        'silver_label': labels,
        'silver_score': combined,
        'cross_encoder_score': ce_scores,
        'semantic_score': semantic_scores[shortlist_indices],
        'bm25_score': bm25_scores[shortlist_indices],
    }).sort_values(['silver_label', 'silver_score', 'candidate_id'], ascending=[False, False, True])

    print(f'Saving {len(df)} actual-JD silver labels to {output_file}...')
    df.to_csv(output_file, index=False)
    print('Silver Labels generated successfully!')

def run_honeypot_detector(input_file, output_file):
    print(f'Honeypot Detector: Scanning {input_file} for high-confidence impossible profiles...')
    honeypot_ids = []

    for record in load_candidate_records(input_file):
        cid = record['candidate_id']
        profile = record.get('profile', {})
        yoe_months = float(profile.get('years_of_experience', 0) or 0) * 12
        career_history = record.get('career_history', [])
        total_career_months = sum(max(0, int(r.get('duration_months', 0) or 0)) for r in career_history)
        skills = record.get('skills', [])
        expert_skills = [s for s in skills if s.get('proficiency') == 'expert']
        zero_duration_experts = [s for s in expert_skills if int(s.get('duration_months', 0) or 0) == 0]

        anomaly_score = 0.0
        if len(career_history) >= 2 and total_career_months > yoe_months + 84:
            anomaly_score += 1.0
        if len(expert_skills) >= 5 and len(zero_duration_experts) / max(len(expert_skills), 1) >= 0.40:
            anomaly_score += 2.0
        elif len(zero_duration_experts) >= 3:
            anomaly_score += 1.5

        title = profile.get('current_title', '').lower()
        career_text = ' '.join(str(role.get('description', '')) for role in career_history).lower()
        skill_text = ' '.join(str(s.get('name', '')) for s in skills).lower()
        ai_skill_hits = token_count(skill_text, JD_POSITIVE_TERMS)
        career_anchor_hits = token_count(career_text + ' ' + title, JD_POSITIVE_TERMS)
        if ai_skill_hits >= 10 and career_anchor_hits <= 1 and any(t in title for t in BAD_TITLE_TERMS):
            anomaly_score += 1.5
        if compute_profile_fit_score(record) < 0.12 and ai_skill_hits >= 12:
            anomaly_score += 1.0

        if anomaly_score >= 2.5:
            honeypot_ids.append(cid)

    print(f'Detected {len(honeypot_ids)} high-confidence honeypots. Saving flags to {output_file}...')
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(honeypot_ids, f, indent=2)
    print('Honeypot detection complete!')

def build_ltr_matrix(selected_indices, semantic_scores, bm25_scores, features, features_ix,
                     candidate_ids, honeypot_set=None, fit_scores=None):
    selected_indices = np.asarray(selected_indices, dtype=int)
    sem_norm = minmax(semantic_scores)
    bm25_norm = minmax(bm25_scores)
    fit_all = np.zeros(len(candidate_ids), dtype=float) if fit_scores is None else np.asarray(fit_scores, dtype=float)
    honeypot_set = honeypot_set or set()

    X_out = np.zeros((len(selected_indices), 27), dtype=np.float32)
    for row_idx, original_idx in enumerate(selected_indices):
        base = features[original_idx]
        ix = features_ix[original_idx]
        cid = candidate_ids[original_idx]
        X_out[row_idx, 0] = sem_norm[original_idx]
        X_out[row_idx, 1] = bm25_norm[original_idx]
        X_out[row_idx, 2] = fit_all[original_idx]
        X_out[row_idx, 3] = np.clip(base[0] / 100.0, 0.0, 1.0)
        X_out[row_idx, 4] = np.clip(base[3], 0.0, 1.0)
        X_out[row_idx, 5] = np.clip(base[16], 0.0, 1.0) * np.clip(base[3], 0.0, 1.0)
        X_out[row_idx, 6] = min(1.0, np.log1p(base[1] + base[9] + base[10]) / np.log1p(700))
        X_out[row_idx, 7] = np.clip(base[8] / 100.0, 0.0, 1.0)
        X_out[row_idx, 8] = 1.0 - np.clip(base[7] / 90.0, 0.0, 1.0)
        X_out[row_idx, 9] = np.clip(base[11], 0.0, 1.0)
        X_out[row_idx, 10] = np.clip(base[12], 0.0, 1.0)
        X_out[row_idx, 11] = np.mean(np.clip(base[13:16], 0.0, 1.0))
        X_out[row_idx, 12] = np.clip(base[17], 0.0, 1.0)
        salary_mid = (base[18] + base[19]) / 2.0
        X_out[row_idx, 13] = 1.0 - np.clip(abs(salary_mid - 45.0) / 45.0, 0.0, 1.0)
        X_out[row_idx, 14] = min(1.0, np.log1p(base[6]) / np.log1p(500))
        X_out[row_idx, 15] = min(1.0, np.log1p(base[5]) / np.log1p(2000))
        X_out[row_idx, 16] = min(1.0, np.log1p(base[2]) / np.log1p(40))
        X_out[row_idx, 17] = 1.0 if cid in honeypot_set else 0.0
        X_out[row_idx, 18:27] = ix[:9]
    return X_out

def run_train_xgboost(artifacts_dir, model_output_path, jd_path=JOB_DESCRIPTION_PATH, candidates_path=CANDIDATES_PATH):
    print('XGBoost LTR: Loading precomputed artifacts...')
    features = np.load(f'{artifacts_dir}/features.npy')
    features_ix = np.load(f'{artifacts_dir}/features_ix.npy')
    meta = np.load(f'{artifacts_dir}/features_meta.npz')
    candidate_ids = np.array(meta['candidate_ids'])

    labels_df = pd.read_csv(f'{artifacts_dir}/silver_labels.csv')
    label_dict = dict(zip(labels_df['candidate_id'], labels_df['silver_label']))
    labeled_indices = np.array([i for i, cid in enumerate(candidate_ids) if cid in label_dict], dtype=int)
    if len(labeled_indices) == 0:
        raise ValueError('No labeled candidates found; cannot train LTR model.')

    jd_text = read_docx_text(jd_path)
    semantic_scores = compute_semantic_scores_from_jd(artifacts_dir, jd_text)
    bm25_scores, bm25_ids = compute_bm25_scores_from_jd(artifacts_dir, jd_text)
    if list(bm25_ids) != list(candidate_ids):
        raise ValueError('BM25 candidate_id order does not match features_meta candidate_id order.')

    raw_by_id = {r['candidate_id']: r for r in load_candidate_records(candidates_path)}
    fit_scores = np.array([compute_profile_fit_score(raw_by_id[cid]) for cid in candidate_ids], dtype=float)

    honeypot_path = Path(artifacts_dir) / 'honeypot_flags.json'
    honeypot_set = set(json.load(open(honeypot_path))) if honeypot_path.exists() else set()

    X_train = build_ltr_matrix(labeled_indices, semantic_scores, bm25_scores, features, features_ix,
                               candidate_ids, honeypot_set=honeypot_set, fit_scores=fit_scores)
    y = np.array([label_dict[candidate_ids[i]] for i in labeled_indices], dtype=float)
    groups = [len(X_train)]

    print(f'Training XGBoost LTR model on actual-JD labels (shape: {X_train.shape})...')
    dtrain = xgb.DMatrix(X_train, label=y)
    dtrain.set_group(groups)
    params = {
        'objective': 'rank:ndcg',
        'eval_metric': 'ndcg@10',
        'eta': 0.06,
        'max_depth': 5,
        'subsample': 0.90,
        'colsample_bytree': 0.90,
        'min_child_weight': 3,
        'seed': 42,
        'verbosity': 1,
    }
    bst = xgb.train(params, dtrain, num_boost_round=120)
    print(f'Saving trained LTR booster to {model_output_path}...')
    bst.save_model(model_output_path)
    print('XGBoost model trained successfully!')


## 5. Phase 3 & 4: Live Retrieval, XGBoost LTR, and Cross-Encoder Re-ranking
Ported from `src/rank.py`. This reads the Job Description, embeds it, computes BM25 and BGE similarities, merges rankings using RRF, runs the XGBoost LTR model, applies a Cross-Encoder over the top 50 candidates, outputs the final formatted reasonings, and enforces format constraints.


In [ ]:
def cosine_similarity(vec, matrix):
    dot_product = np.dot(matrix, vec.T).squeeze()
    norm_vec = np.linalg.norm(vec)
    norm_matrix = np.linalg.norm(matrix, axis=1)
    norm_matrix[norm_matrix == 0] = 1e-10
    if norm_vec == 0:
        norm_vec = 1e-10
    return dot_product / (norm_matrix * norm_vec)

def rrf_score(rank_list_1, rank_list_2, k=60):
    return (1.0 / (k + rank_list_1)) + (1.0 / (k + rank_list_2))

def extract_text_from_docx(file_path):
    try:
        return read_docx_text(file_path)
    except Exception as e:
        print(f'Warning: Failed to read JD from {file_path}: {e}')
        return ''

def generate_verified_reasoning(candidate_profile, rank, jd_text=None):
    profile = candidate_profile.get('profile', {})
    signals = candidate_profile.get('redrob_signals', {})
    title = profile.get('current_title', profile.get('job_title', 'Candidate'))
    yoe = float(profile.get('years_of_experience', 0.0) or 0.0)
    company = profile.get('current_company', 'current company')
    industry = profile.get('current_industry', 'industry')
    location = profile.get('location', 'unknown location')

    skills = [s.get('name', '') for s in candidate_profile.get('skills', [])]
    skill_text = ' '.join(skills).lower()
    matched_skills = [s for s in skills if token_count(s.lower(), JD_POSITIVE_TERMS) > 0][:4]
    if not matched_skills:
        matched_skills = skills[:3]
    matched_skills_str = ', '.join(matched_skills) if matched_skills else 'adjacent engineering skills'

    career_text = ' '.join(str(role.get('description', '')) for role in candidate_profile.get('career_history', [])).lower()
    career_hits = [term for term in JD_POSITIVE_TERMS if term in career_text][:3]
    career_phrase = ', '.join(career_hits) if career_hits else 'production engineering experience'

    response = signals.get('recruiter_response_rate', 0)
    notice = signals.get('notice_period_days', 0)
    github = signals.get('github_activity_score', -1)
    saved = signals.get('saved_by_recruiters_30d', 0)
    relocate = signals.get('willing_to_relocate', False)
    mode = signals.get('preferred_work_mode', 'flexible')

    concerns = []
    if yoe < 5:
        concerns.append('slightly below the preferred 5-9 year band')
    elif yoe > 9:
        concerns.append('above the preferred 5-9 year band')
    if notice and notice > 60:
        concerns.append(f'{notice}-day notice period')
    if response < 0.35:
        concerns.append(f'{int(response * 100)}% recruiter response rate')
    if any(term in title.lower() for term in BAD_TITLE_TERMS):
        concerns.append('current title is adjacent rather than directly AI-focused')

    if rank <= 10:
        opener = 'Strong fit'
    elif rank <= 50:
        opener = 'Relevant fit'
    else:
        opener = 'Lower-ranked fit'

    availability = f'{int(response * 100)}% recruiter response rate, {notice}-day notice, {mode} mode'
    if relocate:
        availability += ', willing to relocate'
    github_phrase = f'GitHub activity score {github}' if github != -1 else 'no linked GitHub signal'
    concern_phrase = f' Watch-outs: {", ".join(concerns)}.' if concerns else ''

    return (
        f'Rank #{rank}: {opener} - {title} at {company} ({industry}) with {yoe:.1f} years in {location}; '
        f'matches the Senior AI Engineer JD through {matched_skills_str} and career evidence around {career_phrase}. '
        f'Hireability signals: {availability}, {saved} recruiter saves, {github_phrase}.{concern_phrase}'
    )

def run_ranking_pipeline(jd_path, candidates_path, enriched_path, artifacts_dir, output_csv_path):
    print('\n' + '='*50 + '\nRUNNING RANKING PIPELINE\n' + '='*50)
    start_time = time.time()

    print('1. Loading precomputed artifacts...')
    features = np.load(f'{artifacts_dir}/features.npy')
    features_ix = np.load(f'{artifacts_dir}/features_ix.npy')
    with open(f'{artifacts_dir}/bm25.pkl', 'rb') as f:
        bm25_data = pickle.load(f)
        bm25_model = bm25_data['bm25_index']
        candidate_ids = np.array(bm25_data['candidate_ids'])
    embeddings = np.load(f'{artifacts_dir}/embeddings.npy')
    with open(f'{artifacts_dir}/honeypot_flags.json', 'r') as f:
        honeypot_set = set(json.load(f))

    print('2. Extracting Job Description text...')
    jd_text = extract_text_from_docx(jd_path)
    if not jd_text.strip():
        raise ValueError('Job description text is empty; refusing to rank against fallback text.')
    print(f'JD Text Length: {len(jd_text)} characters.')

    print('3. Loading raw profiles and computing JD-fit priors...')
    raw_records = load_candidate_records(candidates_path)
    profiles_dict = {record['candidate_id']: record for record in raw_records}
    fit_scores = np.array([compute_profile_fit_score(profiles_dict[cid]) for cid in candidate_ids], dtype=float)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    print('4. Performing dense semantic encoding & search...')
    bge_local_path = Path(artifacts_dir) / 'bge_model'
    bge_model = SentenceTransformer(str(bge_local_path), device=device)
    jd_vector = bge_model.encode([jd_text], convert_to_numpy=True)[0].reshape(1, -1)
    semantic_scores = cosine_similarity(jd_vector, embeddings)
    semantic_ranks = np.empty_like(semantic_scores, dtype=float)
    semantic_ranks[np.argsort(-semantic_scores)] = np.arange(1, len(semantic_scores) + 1)

    print('5. Performing sparse keyword search (BM25)...')
    cleaned_jd = re.sub(r'[^\w\s]', ' ', jd_text.lower())
    jd_tokens = cleaned_jd.split()
    bm25_scores = np.asarray(bm25_model.get_scores(jd_tokens), dtype=float)
    bm25_ranks = np.empty_like(bm25_scores, dtype=float)
    bm25_ranks[np.argsort(-bm25_scores)] = np.arange(1, len(bm25_scores) + 1)

    print('6. Performing RRF retrieval fusion with JD-fit prior and honeypot filtering...')
    fit_ranks = np.empty_like(fit_scores, dtype=float)
    fit_ranks[np.argsort(-fit_scores)] = np.arange(1, len(fit_scores) + 1)
    fused_scores = rrf_score(semantic_ranks, bm25_ranks) + 1.20 * (1.0 / (60 + fit_ranks))
    fused_scores = fused_scores * (0.25 + 0.75 * fit_scores)
    for i, cid in enumerate(candidate_ids):
        if cid in honeypot_set:
            fused_scores[i] = -1.0

    top_n = min(800, len(candidate_ids))
    top_indices = np.argsort(-fused_scores)[:top_n]
    top_ids = candidate_ids[top_indices]

    print('7. Building LTR feature matrix and scoring with XGBoost...')
    X_rank = build_ltr_matrix(top_indices, semantic_scores, bm25_scores, features, features_ix,
                              candidate_ids, honeypot_set=honeypot_set, fit_scores=fit_scores)
    bst = xgb.Booster()
    bst.load_model(f'{artifacts_dir}/xgboost_ltr.model')
    xgb_raw_scores = bst.predict(xgb.DMatrix(X_rank))
    blended_scores = (
        0.52 * minmax(xgb_raw_scores) +
        0.20 * minmax(semantic_scores[top_indices]) +
        0.13 * minmax(bm25_scores[top_indices]) +
        0.15 * fit_scores[top_indices]
    )
    for row_idx, cid in enumerate(top_ids):
        if cid in honeypot_set:
            blended_scores[row_idx] = -1.0

    xgb_sorted_indices = np.argsort(-blended_scores)
    pre_re_rank_indices = xgb_sorted_indices[:min(120, len(xgb_sorted_indices))]

    print('8. Loading enriched candidate texts for Cross-Encoder re-ranking...')
    re_rank_limit = min(50, len(pre_re_rank_indices))
    top_50_indices_of_xgb = pre_re_rank_indices[:re_rank_limit]
    top_50_ids = {top_ids[i] for i in top_50_indices_of_xgb}
    enriched_dict = {}
    with open(enriched_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            record = json.loads(line)
            cid = record['candidate_id']
            if cid in top_50_ids:
                enriched_dict[cid] = record['enriched_text'][:2400]

    candidates_to_ce = []
    for idx in top_50_indices_of_xgb:
        cid = top_ids[idx]
        candidates_to_ce.append((cid, enriched_dict.get(cid, '')))

    print('9. Running Cross-Encoder re-ranker on top 50 candidates...')
    ce_local_path = Path(artifacts_dir) / 'ce_model'
    ce_model = CrossEncoder(str(ce_local_path), device=device)
    pairs = [(jd_text, text) for _, text in candidates_to_ce]
    try:
        ce_scores = ce_model.predict(pairs, batch_size=32, show_progress_bar=True)
    except TypeError:
        ce_scores = ce_model.predict(pairs, batch_size=32)
    ce_scores = np.asarray(ce_scores, dtype=float)
    ce_combo = 0.78 * minmax(ce_scores)
    ce_combo += 0.12 * np.array([fit_scores[np.where(candidate_ids == cid)[0][0]] for cid, _ in candidates_to_ce])
    ce_combo += 0.10 * np.array([minmax(semantic_scores)[np.where(candidate_ids == cid)[0][0]] for cid, _ in candidates_to_ce])

    ce_sorted_sub_indices = np.argsort(-ce_combo)
    sorted_top_50_indices = [top_50_indices_of_xgb[i] for i in ce_sorted_sub_indices]
    remaining_indices = [idx for idx in pre_re_rank_indices[re_rank_limit:] if idx not in sorted_top_50_indices]
    final_merged_indices = list(sorted_top_50_indices) + list(remaining_indices)

    if len(final_merged_indices) < 100:
        seen = set(final_merged_indices)
        for idx in xgb_sorted_indices:
            if idx not in seen:
                final_merged_indices.append(idx)
                seen.add(idx)
            if len(final_merged_indices) >= 100:
                break
    final_merged_indices = final_merged_indices[:100]

    print('10. Formatting normalized scores and fact-specific reasoning...')
    submission_data = []
    output_scores = np.linspace(1.0, 0.01, num=len(final_merged_indices))
    for rank_idx, idx in enumerate(final_merged_indices, start=1):
        original_idx = top_indices[idx]
        cid = candidate_ids[original_idx]
        candidate_profile = profiles_dict.get(cid, {})
        submission_data.append({
            'candidate_id': cid,
            'rank': rank_idx,
            'score': float(output_scores[rank_idx - 1]),
            'reasoning': generate_verified_reasoning(candidate_profile, rank_idx, jd_text),
        })

    if len(submission_data) != 100:
        raise ValueError(f'Expected 100 submission rows, got {len(submission_data)}.')

    df = pd.DataFrame(submission_data)
    df.to_csv(output_csv_path, index=False)
    print(f'\nRanking Pipeline completed in {time.time() - start_time:.2f} seconds!')
    print(f'Saved 100 results to {output_csv_path}')


## 6. End-to-End Orchestrator Run
Run this cell to execute all phases. This will read raw candidate profiles from `/content/India_runs_data_and_ai_challenge/candidates.jsonl` and job description from `/content/India_runs_data_and_ai_challenge/job_description.docx`, perform narrative enrichment, precompute all retrieval indices and dense embeddings, train an XGBoost LTR model, execute hybrid reciprocal rank fusion, apply Deep Cross-Encoder re-ranking, and output the final predictions to `/content/submission.csv`.


In [ ]:
def run_end_to_end_pipeline():
    print('Starting End-to-End Redrob AI Pipeline execution...\n')
    overall_start = time.time()
    
    # 1. Narrative Synthesis
    run_phase1_enrichment(CANDIDATES_PATH, ENRICHED_CANDIDATES_PATH)
    
    # 2. Precomputations
    run_track_a_bm25(ENRICHED_CANDIDATES_PATH, BM25_INDEX_PATH)
    run_track_b_embeddings(ENRICHED_CANDIDATES_PATH, EMBEDDINGS_PATH, batch_size=128)
    run_track_c_features(CANDIDATES_PATH, FEATURES_PATH)
    run_interaction_builder(FEATURES_PATH, FEATURES_IX_PATH)
    run_silver_label_generator(FEATURES_META_PATH, SILVER_LABELS_PATH)
    run_honeypot_detector(CANDIDATES_PATH, HONEYPOT_FLAGS_PATH)
    
    # 3. LTR Model Training
    run_train_xgboost(ARTIFACTS_DIR, XGBOOST_LTR_MODEL_PATH)
    
    # 4. Live Retrieval and Deep Re-ranking
    run_ranking_pipeline(
        jd_path=JOB_DESCRIPTION_PATH,
        candidates_path=CANDIDATES_PATH,
        enriched_path=ENRICHED_CANDIDATES_PATH,
        artifacts_dir=ARTIFACTS_DIR,
        output_csv_path=OUTPUT_SUBMISSION_PATH
    )
    
    print(f'\nAll steps completed successfully in {time.time() - overall_start:.2f} seconds!')
    print(f'Final output file is located at: {OUTPUT_SUBMISSION_PATH}')

if __name__ == '__main__':
    run_end_to_end_pipeline()
